# Final Project — Step 3: Gold Layer
## Source: local Silver Delta | Target: local Gold Delta

3 KPI tables:
- `gold_daily_metrics` → 1 row per day
- `gold_hourly_patterns` → 1 row per (hour × weekend)
- `gold_zone_performance` → 1 row per pickup zone

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("final-project-gold")
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.0.0")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

print(f"Spark version: {spark.version}")
print("✓ Spark session created with Delta support")

Spark version: 3.5.0
✓ Spark session created with Delta support


In [2]:
# Paths
BASE_PATH        = "/workspace/output/final_project"
SILVER_PATH      = f"{BASE_PATH}/silver/yellow"
GOLD_DAILY_PATH  = f"{BASE_PATH}/gold/daily_metrics"
GOLD_HOURLY_PATH = f"{BASE_PATH}/gold/hourly_patterns"
GOLD_ZONE_PATH   = f"{BASE_PATH}/gold/zone_performance"

print(f"Source : {SILVER_PATH}")
print(f"Targets:")
print(f"  {GOLD_DAILY_PATH}")
print(f"  {GOLD_HOURLY_PATH}")
print(f"  {GOLD_ZONE_PATH}")

Source : /workspace/output/final_project/silver/yellow
Targets:
  /workspace/output/final_project/gold/daily_metrics
  /workspace/output/final_project/gold/hourly_patterns
  /workspace/output/final_project/gold/zone_performance


In [3]:
df_silver = spark.read.format("delta").load(SILVER_PATH)
print(f"Silver record count: {df_silver.count():,}")

Silver record count: 2,723,734


In [11]:
# KPI Table 1: gold_daily_metrics
# Grain: 1 row per pickup_date

from pyspark.sql.functions import (
    to_date, col, count, sum as spark_sum,
    avg, round as spark_round, countDistinct
)

df_daily = (
    df_silver
    .withColumn("pickup_date", to_date("tpep_pickup_datetime"))
    .groupBy("pickup_date")
    .agg(
        count("*").alias("total_trips"),
        spark_round(spark_sum("total_amount"), 2).alias("total_revenue"),
        spark_round(avg("total_amount"), 2).alias("avg_fare"),
        spark_round(avg("trip_distance"), 2).alias("avg_distance_miles"),
        spark_round(avg("trip_duration_minutes"), 2).alias("avg_duration_min"),
        spark_round(spark_sum("tip_amount"), 2).alias("total_tips"),
        spark_round(avg("tip_amount") / avg("fare_amount") * 100, 2).alias("avg_tip_pct"),
        spark_sum("passenger_count").alias("total_passengers")
    )
    .orderBy("pickup_date")
)

print(f"Daily metrics rows: {df_daily.count()}")
df_daily.show(10, truncate=False)

Daily metrics rows: 35
+-----------+-----------+-------------+--------+------------------+----------------+----------+-----------+----------------+
|pickup_date|total_trips|total_revenue|avg_fare|avg_distance_miles|avg_duration_min|total_tips|avg_tip_pct|total_passengers|
+-----------+-----------+-------------+--------+------------------+----------------+----------+-----------+----------------+
|2002-12-31 |1          |10.5         |10.5    |0.63              |6.03            |0.0       |0.0        |1               |
|2009-01-01 |3          |127.69       |42.56   |7.44              |27.62           |0.0       |0.0        |4               |
|2023-12-31 |10         |224.62       |22.46   |2.6               |10.16           |26.27     |18.23      |19              |
|2024-01-01 |67408      |2104286.02   |31.22   |4.32              |16.81           |251088.85 |16.95      |105719          |
|2024-01-02 |70485      |2182468.69   |30.96   |4.19              |17.04           |254941.97 |16.92  

In [5]:
(
    df_daily.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_DAILY_PATH)
)
print(f"✓ Daily metrics written: {GOLD_DAILY_PATH}")

✓ Daily metrics written: /workspace/output/final_project/gold/daily_metrics


In [6]:
# KPI Table 2: gold_hourly_patterns
# Grain: 1 row per (pickup_hour, is_weekend)

df_hourly = (
    df_silver
    .groupBy("pickup_hour", "is_weekend")
    .agg(
        count("*").alias("total_trips"),
        spark_round(avg("total_amount"), 2).alias("avg_fare"),
        spark_round(avg("trip_distance"), 2).alias("avg_distance_miles"),
        spark_round(avg("trip_duration_minutes"), 2).alias("avg_duration_min"),
        spark_round(avg("avg_speed_mph"), 2).alias("avg_speed_mph"),
        spark_round(spark_sum("total_amount"), 2).alias("total_revenue")
    )
    .orderBy("is_weekend", "pickup_hour")
)

print(f"Hourly patterns rows: {df_hourly.count()}")
df_hourly.show(10, truncate=False)

Hourly patterns rows: 48
+-----------+----------+-----------+--------+------------------+----------------+-------------+-------------+
|pickup_hour|is_weekend|total_trips|avg_fare|avg_distance_miles|avg_duration_min|avg_speed_mph|total_revenue|
+-----------+----------+-----------+--------+------------------+----------------+-------------+-------------+
|0          |false     |35364      |32.58   |4.71              |15.61           |18.96        |1152013.57   |
|1          |false     |19306      |29.72   |4.09              |15.03           |17.35        |573714.82    |
|2          |false     |12923      |27.69   |3.66              |15.04           |15.36        |357893.82    |
|3          |false     |8871       |30.05   |4.25              |14.81           |16.78        |266580.07    |
|4          |false     |6957       |36.62   |5.65              |17.97           |20.58        |254776.12    |
|5          |false     |12900      |37.07   |6.08              |17.23           |19.85        |

In [7]:
(
    df_hourly.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_HOURLY_PATH)
)
print(f"✓ Hourly patterns written: {GOLD_HOURLY_PATH}")

✓ Hourly patterns written: /workspace/output/final_project/gold/hourly_patterns


In [8]:
# KPI Table 3: gold_zone_performance
# Grain: 1 row per pickup zone (PULocationID)

from pyspark.sql.functions import desc

df_zone = (
    df_silver
    .groupBy("PULocationID")
    .agg(
        count("*").alias("total_trips"),
        spark_round(spark_sum("total_amount"), 2).alias("total_revenue"),
        spark_round(avg("total_amount"), 2).alias("avg_fare"),
        spark_round(avg("trip_distance"), 2).alias("avg_distance_miles"),
        spark_round(avg("tip_amount"), 2).alias("avg_tip"),
        countDistinct("DOLocationID").alias("unique_destinations")
    )
    .orderBy(desc("total_revenue"))
)

print(f"Zone performance rows: {df_zone.count()}")
print("\nTop 10 zones by revenue:")
df_zone.show(10, truncate=False)

Zone performance rows: 253

Top 10 zones by revenue:
+------------+-----------+-------------+--------+------------------+-------+-------------------+
|PULocationID|total_trips|total_revenue|avg_fare|avg_distance_miles|avg_tip|unique_destinations|
+------------+-----------+-------------+--------+------------------+-------+-------------------+
|132         |136950     |1.107762302E7|80.89   |15.85             |9.19   |258                |
|138         |86563      |5746461.95   |66.38   |9.71              |8.8    |256                |
|161         |134967     |3234973.77   |23.97   |2.32              |3.17   |224                |
|237         |135457     |2672977.59   |19.73   |1.7               |2.64   |211                |
|230         |98960      |2658919.87   |26.87   |2.96              |3.39   |225                |
|236         |128098     |2581849.58   |20.16   |1.84              |2.7    |200                |
|186         |99455      |2396573.87   |24.1    |2.29              |3.14  

In [9]:
(
    df_zone.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_ZONE_PATH)
)
print(f"✓ Zone performance written: {GOLD_ZONE_PATH}")

✓ Zone performance written: /workspace/output/final_project/gold/zone_performance


In [10]:
# Final verification — read each Gold table back and confirm row counts

for name, path in [
    ("daily_metrics",    GOLD_DAILY_PATH),
    ("hourly_patterns",  GOLD_HOURLY_PATH),
    ("zone_performance", GOLD_ZONE_PATH),
]:
    df_check = spark.read.format("delta").load(path)
    cnt  = df_check.count()
    cols = len(df_check.columns)
    print(f"  {name:20s} — {cnt:>5} rows, {cols} columns")

print("\n✓ Gold layer complete — ready for Airflow + Power BI")

  daily_metrics        —    35 rows, 9 columns
  hourly_patterns      —    48 rows, 8 columns
  zone_performance     —   253 rows, 7 columns

✓ Gold layer complete — ready for Airflow + Power BI
